In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('C:/Docs/Healthcare-Payment-Integrity/data/PGYR2024_P06302026_06032026/OP_DTL_GNRL_PGYR2024_P06302026_06032026.csv',
                       nrows=0)
print(df.columns.to_list())

In [ ]:
req_cols = [
    # bookkeeping
    'Record_ID', 'Change_Type', 'Program_Year',
    'Payment_Publication_Date', 'Dispute_Status_for_Publication',
    # who was paid
    'Covered_Recipient_Type', 'Covered_Recipient_Profile_ID',
    'Covered_Recipient_NPI', 'Covered_Recipient_First_Name',
    'Covered_Recipient_Last_Name',
    # where
    'Recipient_City', 'Recipient_State', 'Recipient_Zip_Code',
    # what kind of provider
    'Covered_Recipient_Primary_Type_1', 'Covered_Recipient_Specialty_1',
    # who paid
    'Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID',
    'Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name',
    # the payment (signal 1)
    'Total_Amount_of_Payment_USDollars', 'Date_of_Payment',
    'Number_of_Payments_Included_in_Total_Amount',
    'Form_of_Payment_or_Transfer_of_Value',
    'Nature_of_Payment_or_Transfer_of_Value',
    'Physician_Ownership_Indicator',
    # product (slot 1 only)
    'Indicate_Drug_or_Biological_or_Device_or_Medical_Supply_1',
    'Product_Category_or_Therapeutic_Area_1',
    'Name_of_Drug_or_Biological_or_Device_or_Medical_Supply_1',
]

In [ ]:
payments = pd.read_csv('C:/Docs/Healthcare-Payment-Integrity/data/PGYR2024_P06302026_06032026/OP_DTL_GNRL_PGYR2024_P06302026_06032026.csv',
                       chunksize=500_000, usecols=req_cols, dtype=str, na_filter=False)

pieces =[]

for chunk in payments:
    in_scope = chunk['Recipient_State'].isin(['MA','FL'])
    filtered = chunk[in_scope]

    pieces.append(filtered)

opp_scope = pd.concat(pieces, ignore_index=True)
print(len(opp_scope))
print(opp_scope['Recipient_State'].value_counts())
print(opp_scope['Program_Year'].value_counts())

In [ ]:
opp_scope[req_cols].to_parquet('C:/Docs/Healthcare-Payment-Integrity/data/Processed/open_payments_general_2024.parquet', index=False)

In [ ]:
df = pd.read_parquet('C:/Docs/Healthcare-Payment-Integrity/data/Processed/open_payments_general_2024.parquet')
print(df)

In [ ]:
print(df['Covered_Recipient_Type'].value_counts())

In [ ]:
print(df['Covered_Recipient_NPI'].value_counts().head(10))

In [ ]:
blank_npi = df['Covered_Recipient_NPI'] == ""
print(df[blank_npi]['Covered_Recipient_Type'].value_counts())

In [ ]:
print(df[blank_npi]['Covered_Recipient_Profile_ID'])

In [ ]:
blank_profile_id = df[blank_npi]['Covered_Recipient_Profile_ID'] == ""
print(blank_profile_id.sum())

In [ ]:
print(df['Record_ID'].duplicated().sum())

print(df['Nature_of_Payment_or_Transfer_of_Value'].value_counts())

null_amnt = df['Total_Amount_of_Payment_USDollars'] == ""
print(null_amnt.sum())

negative_amnt = df['Total_Amount_of_Payment_USDollars'].str.startswith('-')
print(negative_amnt.sum())